# Neural Networks with PyTorch

## What you'll learn
- What a neural network is, in plain terms
- PyTorch basics: tensors and automatic gradients
- Building a small network with `nn.Sequential`
- Writing a training loop: forward pass, loss, backward pass, update
- Evaluating the network and comparing it with scikit-learn

PyTorch is installed with `pip install torch` (it is in `requirements.txt`).

## Neural networks in one paragraph

A neural network is layers of simple units. Each unit takes a weighted sum of its inputs
and passes it through a small bend (an **activation**, such as ReLU). Stacking layers lets
the network learn curved, interacting patterns that a straight-line model cannot. Training
means nudging all the weights, a little at a time, to reduce the error (the **loss**).
The direction of each nudge comes from the **gradient**, which PyTorch computes for you.

In [ ]:
import torch

torch.manual_seed(42)   # repeatable results
print("PyTorch", torch.__version__)

## Tensors and gradients

A **tensor** is PyTorch's array, like a NumPy array that can also track how it was
computed. With `requires_grad=True`, PyTorch records the maths and `backward()` works out
the gradient: how much the result changes when the input changes.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 1) ** 2        # smallest when w is 1
loss.backward()            # compute d(loss)/dw
print("loss:", loss.item(), " gradient:", w.grad.item())   # 2 * (3 - 1) = 4

A positive gradient says "decrease `w` to reduce the loss". Training repeats that nudge
for every weight in the network, thousands of times.

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Neural networks need scaled inputs. Fit the scaler on training data only.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype("float32")
X_test_s = scaler.transform(X_test).astype("float32")
print("inputs per account:", X_train_s.shape[1])

## From NumPy to tensors, in batches

A `TensorDataset` pairs inputs with targets, and a `DataLoader` serves them in shuffled
**batches** (here 64 accounts at a time), which is how networks are trained.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

X_test_t = torch.tensor(X_test_s)

## The network

Two hidden layers of 32 and 16 units with ReLU, and one output unit. The output is a raw
score (a **logit**); the loss function turns it into a probability internally, which is
more numerically stable than adding a sigmoid layer ourselves.

In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Linear(X_train_s.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
loss_fn = nn.BCEWithLogitsLoss()                       # binary cross-entropy for yes/no
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)

## The training loop

For each batch: predict (**forward pass**), measure the loss, compute gradients
(**backward pass**), and let the optimiser update the weights. One pass over all the
training data is an **epoch**.

In [ ]:
history = []
for epoch in range(40):
    model.train()
    total = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()                          # clear last batch's gradients
        logits = model(inputs).squeeze(1)              # forward pass
        loss = loss_fn(logits, targets)                # how wrong are we?
        loss.backward()                                # gradients
        optimizer.step()                               # update weights
        total += loss.item() * len(targets)
    history.append(total / len(train_data))
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1:2d}  training loss {history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, len(history) + 1), history)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("PyTorch training loss")
plt.show()

## Evaluate

Switch to evaluation mode, turn off gradient tracking (faster, and not needed to predict),
and turn logits into probabilities with a sigmoid.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

model.eval()
with torch.no_grad():
    probability = torch.sigmoid(model(X_test_t).squeeze(1)).numpy()

predicted = (probability >= 0.5).astype(int)
print(f"Neural network  accuracy {accuracy_score(y_test, predicted):.1%}  "
      f"ROC AUC {roc_auc_score(y_test, probability):.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic = LogisticRegression().fit(X_train_s, y_train)
p = logistic.predict_proba(X_test_s)[:, 1]
print(f"Logistic regr.  accuracy {accuracy_score(y_test, (p >= 0.5).astype(int)):.1%}  "
      f"ROC AUC {roc_auc_score(y_test, p):.3f}")

On a small, tidy table like this, a neural network usually does no better than logistic
regression (here it scores slightly lower). Neural networks shine on large datasets and on images, text, and
sound. For everyday tabular business data, try the simpler scikit-learn models first.

## Practice

1. Train for 100 epochs instead of 40. Does the test ROC AUC improve, or start to fall?
2. Make the network wider (64 and 32 units). What happens to training loss and test score?
3. Add `nn.Dropout(0.2)` after each ReLU (it randomly switches off units during training to
   reduce overfitting) and compare.
4. Save the trained weights with `torch.save(model.state_dict(), "models/late_payment_nn.pt")`
   and load them into a fresh model of the same shape with
   `torch.load(..., weights_only=True)` (which loads only numbers, never code).

In [ ]:
# Your practice code here

## Summary

- A neural network stacks layers of weighted sums and activations.
- Tensors are PyTorch arrays; `backward()` computes gradients automatically.
- `nn.Sequential` builds a network; `BCEWithLogitsLoss` and `Adam` train a yes/no model.
- The training loop is: zero gradients, forward, loss, backward, step.
- Use `model.eval()` and `torch.no_grad()` to predict; compare with a simple baseline model.

## Practice Solutions

In [ ]:
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score


df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Neural networks need scaled inputs. Fit the scaler on training data only.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype("float32")
X_test_s = scaler.transform(X_test).astype("float32")

train_data = TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train, dtype=torch.float32))
X_test_t = torch.tensor(X_test_s)


def train(model, epochs):
    """Train a network and return its test ROC AUC."""
    torch.manual_seed(42)
    loader = DataLoader(train_data, batch_size=64, shuffle=True)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for _ in range(epochs):
        model.train()
        for inputs, targets in loader:
            optimizer.zero_grad()
            loss_fn(model(inputs).squeeze(1), targets).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        return roc_auc_score(y_test, torch.sigmoid(model(X_test_t).squeeze(1)).numpy())


def network(h1=32, h2=16, dropout=0.0):
    torch.manual_seed(42)
    return nn.Sequential(nn.Linear(X_train_s.shape[1], h1), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(h2, 1))


# 1.
for epochs in [40, 100]:
    print(epochs, "epochs:", round(train(network(), epochs), 3))

# 2.
print("wider:", round(train(network(64, 32), 40), 3))

# 3.
print("with dropout:", round(train(network(dropout=0.2), 40), 3))

# 4.
model = network()
train(model, 40)
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/late_payment_nn.pt")
fresh = network()
fresh.load_state_dict(torch.load("models/late_payment_nn.pt", weights_only=True))
fresh.eval()
with torch.no_grad():
    print("reloaded model ROC AUC:",
          round(roc_auc_score(y_test, torch.sigmoid(fresh(X_test_t).squeeze(1)).numpy()), 3))